<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-06-function-calling/lesson-6.3-parallel-tools/practice/GCP_Capstone_6.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 6.3 — Parallel Calls & Built-in Tools

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install the unified `google-genai` SDK, authenticate with Application Default Credentials, initialize the Vertex client (`enterprise=True`), and define the DocuMind functions reused from Lessons 6.1–6.2. Run this cell first — every exercise below depends on `client`, `types`, and `TOOLS`.

In [ ]:
%%bash
pip install -q google-genai

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID,
                      location='us-central1')

# Reuse DocuMind functions from Lesson 6.1-6.2
def search_documents(query: str, doc_type: str = 'all') -> dict:
    """Search DocuMind internal documents."""
    mock = {'legal': [{'id': 'D-01', 'title': 'Privacy Policy v3', 'pages': 12}]}
    results = mock.get(doc_type, [{'id': 'D-99', 'title': f'Result for: {query}', 'pages': 5}])
    return {'documents': results, 'total': len(results)}

def calculate_processing_cost(num_documents: int, total_pages: int, processing_type: str = 'standard') -> dict:
    """Estimate document processing cost."""
    rates = {'standard': 0.05, 'priority': 0.12, 'bulk': 0.03}
    cost = total_pages * rates.get(processing_type, 0.05)
    return {'cost_usd': round(cost, 2), 'cost_inr': round(cost * 85, 2)}

def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get RAG pipeline usage statistics."""
    return {'metric': metric, 'period': f'last {days} days', 'value': 1247}

TOOLS = [search_documents, calculate_processing_cost, get_usage_stats]
print('Ready')

## Exercise 1: Google Search Grounding

**Difficulty:** Easy

Enable Google Search. Ask about regulations. Print grounding_metadata sources.

1. types.Tool(google_search=types.GoogleSearch())
2. Ask about external regulations
3. Print web_search_queries and grounding_chunks

In [ ]:
# Google Search: model queries the web automatically
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What are the latest data protection regulations in India for 2026?',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())])
)
print('=== Google Search Grounded Response ===')
print(response.text[:300])

# Check grounding metadata
metadata = response.candidates[0].grounding_metadata
if metadata:
    print(f'\nSearch queries: {metadata.web_search_queries}')
    if metadata.grounding_chunks:
        for chunk in metadata.grounding_chunks[:3]:
            print(f'  Source: {chunk.web.title}')
            print(f'  URL: {chunk.web.uri}')

## Exercise 2: Code Execution

**Difficulty:** Easy

Enable Code Execution. Ask for a calculation. Print generated code and output.

1. types.Tool(code_execution=types.ToolCodeExecution)
2. Ask a math/stats question
3. Print executable_code and code_execution_result parts

In [ ]:
# Code Execution: Gemini generates + runs Python
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Calculate compound interest on Rs 10,00,000 at 8.5% '
             'for 5 years compounded quarterly. Show the formula and result.',
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)])
)

print('=== Code Execution Response ===')
for part in response.candidates[0].content.parts:
    if part.executable_code:
        print(f'Generated code:\n{part.executable_code.code}\n')
    if part.code_execution_result:
        print(f'Execution result: {part.code_execution_result.output}')
    if part.text:
        print(f'Explanation: {part.text[:200]}')

## Exercise 3: Detect Parallel Calls

**Difficulty:** Easy

Trigger 2 independent function calls. Verify len(response.function_calls) == 2.

1. Query with two independent parts ("search X AND show Y stats")
2. Disable auto function calling
3. Check function_calls list length

In [ ]:
# Trigger parallel calls with an independent multi-part query
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Search for our legal documents AND show this week query stats',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
)

print('=== Parallel Calls ===')
if response.function_calls:
    print(f'Number of calls: {len(response.function_calls)}')
    for fc in response.function_calls:
        print(f'  {fc.name}({dict(fc.args)}) id={fc.id}')
else:
    print('No function calls (model responded with text)')

## Exercise 4: Concurrent Execution

**Difficulty:** Medium

Execute 3 parallel calls with ThreadPoolExecutor. Measure time saved vs sequential.

1. ThreadPoolExecutor with max_workers=N
2. Submit all calls, collect results with matching ids
3. Compare elapsed time: parallel vs sequential

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def execute_parallel_sync(function_calls, functions):
    """Execute multiple function calls concurrently."""
    start = time.time()
    results = [None] * len(function_calls)
    with ThreadPoolExecutor(max_workers=len(function_calls)) as pool:
        futures = {
            pool.submit(functions[fc.name], **fc.args): i
            for i, fc in enumerate(function_calls)
        }
        for future in futures:
            i = futures[future]
            fc = function_calls[i]
            try:
                result = future.result(timeout=15)
                results[i] = types.Part.from_function_response(
                    name=fc.name, response={'result': result})
            except Exception as e:
                results[i] = types.Part.from_function_response(
                    name=fc.name, response={'error': str(e)})
    elapsed = time.time() - start
    print(f'Parallel execution: {elapsed:.2f}s for {len(function_calls)} calls')
    return results

# Test with the parallel calls from Exercise 3
if response.function_calls and len(response.function_calls) > 1:
    FUNCTIONS = {'search_documents': search_documents,
                 'calculate_processing_cost': calculate_processing_cost,
                 'get_usage_stats': get_usage_stats}
    result_parts = execute_parallel_sync(
        list(response.function_calls), FUNCTIONS)
    print(f'Got {len(result_parts)} results')
    for rp in result_parts:
        print(f'  {rp.function_response.name}: {rp.function_response.response}')

## Exercise 5: Google Search + Custom

**Difficulty:** Medium

Combine Google Search with search_documents. Ask a compliance comparison question.

1. Both tool types in tools list
2. System instruction for routing
3. Ask: "How does our policy compare to GDPR?"

In [ ]:
# Combine Google Search with custom document search
chat = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(google_search=types.GoogleSearch()),
            types.Tool(function_declarations=[
                types.FunctionDeclaration(
                    name='search_documents',
                    description='Search internal company documents by keyword.',
                    parameters={'type': 'object',
                                'properties': {'query': {'type': 'string'},
                                               'doc_type': {'type': 'string', 'enum': ['legal', 'invoice', 'all']}},
                                'required': ['query']})
            ]),
        ],
        system_instruction='You are DocuMind AI. Use search_documents for '
                           'internal company documents. Use Google Search for '
                           'external regulations and laws.'
    )
)

# Test: internal question
r1 = chat.send_message('Find our privacy policy')
print(f'Internal query: {r1.text[:150]}...' if len(r1.text) > 150 else f'Internal: {r1.text}')

# Test: external question
r2 = chat.send_message('What are the latest GDPR updates?')
print(f'\nExternal query: {r2.text[:150]}...' if len(r2.text) > 150 else f'\nExternal: {r2.text}')

## Exercise 6: Partial Failure

**Difficulty:** Medium

Make one of 3 parallel calls fail. Verify model explains partial results + error.

1. Make one function raise an exception
2. Send error FunctionResponse for failed call
3. Verify model presents successful results + explains failure

In [ ]:
# Reuse execute_parallel_sync from Exercise 4 — it already sends an
# {'error': ...} FunctionResponse when a function raises. Here we force one
# of three parallel calls to fail, then feed all results back to the model.

def get_usage_stats_flaky(metric: str, days: int = 7) -> dict:
    """Simulate an unavailable analytics service."""
    raise RuntimeError('analytics service unavailable (503)')

# 1) Get the model to request 3 independent calls
fail_query = ('Search our legal documents, estimate the cost to process '
              '200 pages at standard rate, AND show this week query stats')
first = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=fail_query,
    config=types.GenerateContentConfig(
        tools=TOOLS,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
)
print(f'Model requested {len(first.function_calls or [])} calls')

# 2) Execute in parallel — get_usage_stats is wired to the flaky version
FUNCTIONS_FLAKY = {'search_documents': search_documents,
                   'calculate_processing_cost': calculate_processing_cost,
                   'get_usage_stats': get_usage_stats_flaky}
result_parts = execute_parallel_sync(list(first.function_calls), FUNCTIONS_FLAKY)
for rp in result_parts:
    print(f'  {rp.function_response.name}: {rp.function_response.response}')

# 3) Send the mixed success/error results back; model adapts its answer
follow_up = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=[
        types.Content(role='user', parts=[types.Part(text=fail_query)]),
        first.candidates[0].content,
        types.Content(role='user', parts=result_parts),
    ],
    config=types.GenerateContentConfig(tools=TOOLS)
)
print('\n=== Model response with partial failure ===')
print(follow_up.text)

## Exercise 7: Three-Tool Combination

**Difficulty:** Challenge

Combine Google Search + Code Execution + custom functions. Ask a complex multi-source query.

1. All three tool types in one config
2. Ask: "Compare our costs against industry benchmarks and calculate the difference"
3. Verify model routes to appropriate tools

In [ ]:
# Custom function for data + Code Execution for computation
chat2 = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(code_execution=types.ToolCodeExecution),
            *TOOLS,  # Custom functions
        ],
        system_instruction='You are DocuMind AI. Use custom functions to get '
                           'document data. Use Code Execution for calculations '
                           'and statistical analysis on that data.'
    )
)

r = chat2.send_message('How much would it cost to process 500 pages at '
                       'standard rate? Calculate the monthly cost if we '
                       'process this amount weekly.')
print(f'Response: {r.text[:300]}')

In [ ]:
# All three tool types (Google Search + Code Execution + custom) in one config
SYSTEM_PROMPT = '''You are DocuMind AI, a document intelligence assistant.
TOOL ROUTING:
- search_documents: internal company documents
- calculate_processing_cost: cost estimation
- get_usage_stats: pipeline analytics
- Google Search: external regulations, industry benchmarks
- Code Execution: calculations, statistics, charts
Always use the most appropriate tool for each query part.'''

agent_config = types.GenerateContentConfig(
    tools=[
        types.Tool(google_search=types.GoogleSearch()),
        types.Tool(code_execution=types.ToolCodeExecution),
        *TOOLS,
    ],
    system_instruction=SYSTEM_PROMPT
)

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Compare our standard processing cost for 300 pages against '
             'typical industry benchmarks and calculate the difference.',
    config=agent_config)
print('=== Three-tool combined query ===')
tool_used = 'function_call' if r.function_calls else 'text/built-in'
print(f'Tool: {tool_used}')
print(r.text[:400] if r.text else '[function call pending]')

## Exercise 8: DocuMind Multi-Tool Agent

**Difficulty:** Challenge

Build complete agent with system instruction routing. Test 5 queries triggering different tool combinations.

1. System instruction with explicit routing rules
2. All three tool types configured
3. Test: internal, external, calculation, comparison, greeting

In [ ]:
# Complete multi-tool agent (reuses agent_config from Exercise 7)
# Test five routing scenarios: internal, external, calculation, comparison, greeting
test_queries = [
    'Find our legal documents',                              # Custom only
    'What is the current DPDP Act in India?',                # Google Search
    'Calculate 15% tax on Rs 2,50,000',                      # Code Execution
    'Compare our processing cost to industry averages',      # Search + custom + code
    'Hello, how can you help me?',                           # No tools (text)
]

for q in test_queries:
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=q, config=agent_config)
    tool_used = 'function_call' if r.function_calls else 'text/built-in'
    print(f'Q: {q}')
    print(f'  Tool: {tool_used}')
    print(f'  A: {r.text[:100] if r.text else "[function call pending]"}\n')